In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame


class BronzeDimensionLoader:

    def __init__(
        self,
        source_table: str,
        target_table: str
    ):
        self.source_table = source_table
        self.target_table = target_table

    def read_raw(self) -> DataFrame:
        return spark.table(self.source_table)

    def add_audit_columns(self, df: DataFrame) -> DataFrame:

        return (
            df
            .withColumn(
                "_bronze_load_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "_bronze_load_date",
                F.current_date()
            )
            .withColumn(
                "_bronze_source_table",
                F.lit(self.source_table)
            )
        )

    def write_bronze(self, df: DataFrame):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(self.target_table)
        )

    def run(self, transformation_function=None):

        print(f"Source : {self.source_table}")
        print(f"Target : {self.target_table}")

        df = self.read_raw()

        if transformation_function:
            df = transformation_function(df)

        df = self.add_audit_columns(df)

        self.write_bronze(df)

        print(f"Completed: {self.target_table}")

In [0]:
def transform_customer(df):

    return (
        df

        # String cleanup
        .withColumn(
            "customer_name",
            F.initcap(F.trim(F.col("customer_name")))
        )

        .withColumn(
            "email",
            F.lower(F.trim(F.col("email")))
        )

        .withColumn(
            "gender",
            F.initcap(F.trim(F.col("gender")))
        )

        .withColumn(
            "city",
            F.initcap(F.trim(F.col("city")))
        )

        .withColumn(
            "state",
            F.initcap(F.trim(F.col("state")))
        )

        .withColumn(
            "region",
            F.initcap(F.trim(F.col("region")))
        )

        .withColumn(
            "segment",
            F.initcap(F.trim(F.col("segment")))
        )

        # Type conversions
        .withColumn(
            "age",
            F.col("age").cast("int")
        )

        .withColumn(
            "signup_date",
            F.to_date(F.col("signup_date"))
        )

        .withColumn(
            "loyalty_member",
            F.col("loyalty_member").cast("boolean")
        )

        .withColumn(
            "loyalty_points",
            F.col("loyalty_points").cast("int")
        )
    )

In [0]:
customer_loader = BronzeDimensionLoader(
    source_table="retail_demo.raw.customers",
    target_table="retail_demo.bronze.dim_customer"
)

customer_loader.run(
    transformation_function=transform_customer
)

In [0]:
def transform_product(df):

    return (
        df

        .withColumn(
            "product_name",
            F.trim(F.col("product_name"))
        )

        .withColumn(
            "category",
            F.initcap(F.trim(F.col("category")))
        )

        .withColumn(
            "subcategory",
            F.initcap(F.trim(F.col("subcategory")))
        )

        .withColumn(
            "brand",
            F.initcap(F.trim(F.col("brand")))
        )

        .withColumn(
            "product_status",
            F.upper(F.trim(F.col("product_status")))
        )

        .withColumn(
            "tax_code",
            F.upper(F.trim(F.col("tax_code")))
        )

        .withColumn(
            "unit_cost",
            F.col("unit_cost").cast("decimal(18,2)")
        )

        .withColumn(
            "list_price",
            F.col("list_price").cast("decimal(18,2)")
        )

        .withColumn(
            "weight_kg",
            F.col("weight_kg").cast("decimal(10,2)")
        )
    )

In [0]:
product_loader = BronzeDimensionLoader(
    source_table="retail_demo.raw.products",
    target_table="retail_demo.bronze.dim_product"
)

product_loader.run(
    transformation_function=transform_product
)

In [0]:
def transform_campaign(df):

    return (
        df

        .withColumn(
            "campaign_name",
            F.trim(F.col("campaign_name"))
        )

        .withColumn(
            "marketing_channel",
            F.initcap(
                F.trim(F.col("marketing_channel"))
            )
        )

        .withColumn(
            "objective",
            F.initcap(F.trim(F.col("objective")))
        )

        .withColumn(
            "start_date",
            F.to_date(F.col("start_date"))
        )

        .withColumn(
            "end_date",
            F.to_date(F.col("end_date"))
        )

        .withColumn(
            "budget_inr",
            F.col("budget_inr").cast("decimal(18,2)")
        )
    )

In [0]:
campaign_loader = BronzeDimensionLoader(
    source_table="retail_demo.raw.campaigns",
    target_table="retail_demo.bronze.dim_campaign"
)

campaign_loader.run(
    transformation_function=transform_campaign
)

In [0]:
def transform_store(df):

    return (
        df

        .withColumn(
            "store_name",
            F.trim(F.col("store_name"))
        )

        .withColumn(
            "city",
            F.initcap(F.trim(F.col("city")))
        )

        .withColumn(
            "state",
            F.initcap(F.trim(F.col("state")))
        )

        .withColumn(
            "region",
            F.initcap(F.trim(F.col("region")))
        )

        .withColumn(
            "store_format",
            F.initcap(F.trim(F.col("store_format")))
        )

        .withColumn(
            "open_date",
            F.to_date(F.col("open_date"))
        )

        .withColumn(
            "employee_count",
            F.col("employee_count").cast("int")
        )
    )

In [0]:
store_loader = BronzeDimensionLoader(
    source_table="retail_demo.raw.stores",
    target_table="retail_demo.bronze.dim_store"
)

store_loader.run(
    transformation_function=transform_store
)